# Programmatic EDA — 성남시 자영업자 분석

**분석 기간**: 2023.01 ~ 2025.12 | **분석 단위**: 분당구·수정구·중원구

**핵심 컬럼**: 27개 (가맹점 생존, 매출, 자영업자 이동, 금융취약성)

In [ ]:
# ====================================================
# CONFIG BLOCK
# ====================================================
DATA_PATH         = 'master_district_monthly.csv'
TARGET_COL        = None
MISSING_THRESHOLD = 5.0
OUTLIER_THRESHOLD = 1.5
DOMAIN_NOTES      = [
    '2025년 out_self_ent/out_avg_inc 전체 결측 — self_ent_net은 2023-2024만 신뢰',
    '분당구 매출 스케일이 수정·중원구와 수십 배 차이 — 구별 분리 분석 필요',
    'close_rate = mer_close / mer_total * 100 (파생 컬럼)',
    'self_ent_net = in_self_ent - out_self_ent (파생 컬럼)',
    'loan_delinq_rate = over_loan_cnt / econ_cnt (파생 컬럼)'
]
CORE_COLS = [
    'ym', 'gu_nm',
    'mer_total', 'mer_open', 'mer_close', 'mer_net', 'close_rate', 'mer_op_5yp',
    'sales_amt', 'sales_cnt', 'sales_per_merchant',
    'in_self_ent', 'out_self_ent', 'self_ent_net', 'in_avg_inc', 'out_avg_inc',
    'sum_income', 'sum_loan', 'sum_hous_loan', 'new_hous_cnt', 'new_hous_rate',
    'over_loan_cnt', 'loan_delinq_rate', 'score_high', 'score_low', 'econ_cnt',
    'total_inflow'
]
print('Config loaded')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

BASE = '/sessions/hopeful-elegant-davinci/mnt/성남시/'
master = pd.read_csv(BASE + DATA_PATH)
df = master[CORE_COLS].copy()
df['ym'] = df['ym'].astype(str)

print(f'Data loaded: {df.shape[0]} rows x {df.shape[1]} cols')
print(f'Period: {df["ym"].min()} ~ {df["ym"].max()}')
print(f'Districts: {df["gu_nm"].unique()}')

## 1. 결측값 분석

In [ ]:
missing = pd.DataFrame({
    '컬럼': df.columns,
    '결측수': df.isna().sum().values,
    '결측율(%)': (df.isna().mean() * 100).round(2).values,
}).sort_values('결측율(%)', ascending=False)
missing['상태'] = missing['결측율(%)'].apply(
    lambda x: '심각(>30%)' if x > 30 else ('주의(>5%)' if x > MISSING_THRESHOLD else '양호')
)
print(missing[missing['결측수']>0].to_string(index=False))
print('\n[주의] 2025년 전출 데이터(out_self_ent, out_avg_inc) 결측 — self_ent_net 해석 시 주의')

## 2. 구별 핵심 지표 요약

In [ ]:
gu_summary = df.groupby('gu_nm').agg({
    'mer_total': 'mean', 'mer_open': 'mean', 'mer_close': 'mean', 'mer_net': 'mean',
    'close_rate': 'mean', 'sales_amt': 'mean', 'sales_per_merchant': 'mean',
    'in_self_ent': 'mean', 'out_self_ent': 'mean', 'self_ent_net': 'mean',
    'loan_delinq_rate': 'mean', 'sum_income': 'mean'
}).round(2)
print('=== 구별 월평균 핵심 지표 ===')
print(gu_summary.T.to_string())

## 3. 가맹점 생존 트렌드 시각화

In [ ]:
colors = {'분당구': '#2196F3', '수정구': '#FF5722', '중원구': '#4CAF50'}
xticks = list(range(0, 36, 6))
xlabels = sorted(df['ym'].unique())[::6]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Seongnam Merchant Survival Trends (2023-2025)', fontsize=14, fontweight='bold')

for gu, col in colors.items():
    d = df[df['gu_nm']==gu].sort_values('ym')
    x = range(len(d))
    axes[0,0].plot(x, d['mer_open'], color=col, linestyle='-', marker='o', markersize=3, label=f'{gu} Open')
    axes[0,0].plot(x, d['mer_close'], color=col, linestyle='--', marker='s', markersize=3, alpha=0.7)
    axes[0,1].plot(x, d['mer_net'], color=col, marker='o', markersize=4, label=gu)
    axes[1,0].plot(x, d['close_rate'], color=col, marker='o', markersize=4, label=gu)
    axes[1,1].plot(x, d['mer_op_5yp']/d['mer_total']*100, color=col, marker='o', markersize=4, label=gu)

axes[0,1].axhline(0, color='black', linewidth=2, linestyle='--')
for ax, title in zip(axes.flat, 
    ['Open/Close (solid=open, dash=close)', 'Net Change (Open-Close)',
     'Closure Rate (%)', '5yr+ Survival Rate (%)']):
    ax.set_title(title, fontsize=10)
    ax.set_xticks(xticks); ax.set_xticklabels(xlabels, rotation=30, fontsize=8)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(BASE + 'eda_01_merchant_survival.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_01_merchant_survival.png')

## 4. 자영업자 이동 분석 (2023-2024 검증 데이터)

In [ ]:
# 전출 데이터 신뢰 기간만
df_valid = df[df['out_self_ent'].notna()].copy()

print('=== Self-Employed Migration (2023-2024 Verified) ===')
for gu in ['분당구', '수정구', '중원구']:
    d = df_valid[df_valid['gu_nm']==gu]
    avg_net = d['self_ent_net'].mean()
    total_in = d['in_self_ent'].sum()
    total_out = d['out_self_ent'].sum()
    print(f'{gu}: Monthly avg net = {avg_net:.1f}, Cumulative in = {total_in:,.0f}, out = {total_out:,.0f}')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Self-Employed Migration (2023-2024, Verified)', fontsize=14, fontweight='bold')

for gu, col in colors.items():
    d = df_valid[df_valid['gu_nm']==gu].sort_values('ym')
    x = range(len(d))
    axes[0].plot(x, d['self_ent_net'], marker='o', markersize=5, label=gu, color=col)
    axes[1].plot(x, d['in_avg_inc'], color=col, linestyle='-', marker='o', markersize=3, label=f'{gu} In')
    axes[1].plot(x, d['out_avg_inc'], color=col, linestyle='--', marker='s', markersize=3, alpha=0.7)

axes[0].axhline(0, color='black', linewidth=2, linestyle='--')
axes[0].set_title('Net Self-Employed Migration\nNegative = Outflow Dominant', fontsize=11)
axes[1].set_title('Avg Income: Incoming vs Outgoing\n(Solid=In, Dash=Out)', fontsize=11)
for ax in axes:
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(BASE + 'eda_08_migration_verified.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 상관관계 분석

In [ ]:
num_cols = [c for c in CORE_COLS if c not in ['ym', 'gu_nm']]
corr = df[num_cols].corr()

pairs = []
for i in range(len(num_cols)):
    for j in range(i+1, len(num_cols)):
        r = corr.iloc[i, j]
        if abs(r) >= 0.7:
            pairs.append((num_cols[i], num_cols[j], round(r, 3)))

print('Strong correlations (|r| >= 0.7):')
for p in sorted(pairs, key=lambda x: abs(x[2]), reverse=True)[:15]:
    print(f'  {p[0]} <-> {p[1]}: r={p[2]}')

## 6. 도출된 가설

### 가설 1: 구별 자영업 생존 불평등
> 수정구·중원구 자영업자는 분당구 대비 매출은 낮고 폐업률은 높아 구조적 불이익에 처함

### 가설 2: 자영업자 이탈 → 상권 공동화
> 자영업자 지속적 순유출(분당구 月 -748명)은 상권 활력 저하로 이어짐

### 가설 3: 젠트리피케이션 → 자영업자 구조조정
> 주택담보대출·신규주택 취득 증가는 임대료 매개로 자영업자 이탈 가속

### 가설 4: 소득-대출-연체 악순환 (수정·중원구)
> 낮은 매출 → 생계형 대출 → 연체 증가의 취약 구조 존재

**다음 단계**: `stats-basic` → `time-series-analysis` → `ml-unsupervised`